In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import numpy as np
from tqdm import tqdm

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from google.colab import drive

In [ ]:
train_file_path = '/content/drive/MyDrive/Colab Notebooks/Third semester/Fine-Tuning/RuBert-Data/rusentitweet_train.csv'
test_file_path  = '/content/drive/MyDrive/Colab Notebooks/Third semester/Fine-Tuning/RuBert-Data/rusentitweet_test.csv'

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [ ]:
model_name = "unsloth/gemma-3-12b-it-bnb-4bit"
print(f"Загружаем модель: {model_name}...")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        # token = None # Явно указываем отсутствие токена
    )
except Exception as e:
    print("\n!!! КРИТИЧЕСКАЯ ОШИБКА ЗАГРУЗКИ МОДЕЛИ !!!")
    print("Вероятная причина: Модель Gemma требует принятия лицензии (Gated Repo).")
    print("Текст ошибки:", e)
    raise e

Загружаем модель: unsloth/gemma-3-12b-it-bnb-4bit...
==((====))==  Unsloth 2026.1.3: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.81G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


In [ ]:
dataset = load_dataset("csv", data_files={"train": train_file_path, "test": test_file_path})

def formatting_prompts_func(examples):
    texts = []
    sys_prompt = "Ты эксперт-политолог. Проанализируй тональность текста."

    for text, label in zip(examples["text"], examples["label"]):
        messages = [
            {"role": "user", "content": f"{sys_prompt}\n\nТекст: {text}"},
            {"role": "assistant", "content": f"Тональность: {label}"}
        ]
        text_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text_formatted)
    return {"text": texts}

train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10713 [00:00<?, ? examples/s]

In [ ]:
print("Повторная попытка токенизации с явным указанием типа данных...")

def tokenize_function(examples):
    # 1. Жесткое приведение к строкам (защита от None и не-строковых объектов)
    # Если вдруг проскочил None, он станет пустой строкой
    texts = [str(t) if t is not None else "" for t in examples["text"]]

    # 2. Вызов токенизатора с ЯВНЫМ аргументом text=
    # Это критически важно для мультимодальных процессоров Gemma 3
    model_inputs = tokenizer(
        text = texts, # <--- ВОТ ЗДЕСЬ БЫЛА ПРОБЛЕМА. Указываем явно.
        max_length = max_seq_length,
        truncation = True,
        padding = False, # Паддинг оставим коллатору
        add_special_tokens = True,
    )

    # Копируем input_ids в labels для обучения с учителем
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

source_dataset = clean_train_dataset if 'clean_train_dataset' in locals() else dataset["train"]

tokenized_dataset = source_dataset.map(
    tokenize_function,
    batched = True,
    remove_columns = source_dataset.column_names,
    desc = "Running fixed tokenizer",
)



In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, padding = True)

trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    tokenizer = tokenizer, # Важно передать его сюда для сохранения
    data_collator = data_collator,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

print("\nФинальный запуск обучения...")
trainer_stats = trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

# Функция для получения чистого ответа от модели
def predict_sentiment(text):
    messages = [
        {"role": "user", "content": "Ты эксперт-политолог. Проанализируй тональность текста.\n\nТекст: " + text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Добавляет токен начала ответа
        return_tensors = "pt",
    ).to("cuda")

    # Генерация
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 10, # Нам нужно только одно слово (POSITIVE/NEGATIVE...)
        use_cache = True,
        temperature = 0.1, # Почти детерминированный вывод
        top_k = 50,
        top_p = 0.95,
    )

    # Декодинг (убираем сам промпт из ответа)
    decoded = tokenizer.batch_decode(outputs)
    # Парсинг ответа (берем текст после "assistant")
    response = decoded[0].split("assistant\n")[-1].replace("<eos>", "").strip()
    return response

In [ ]:
print("Начинаем генерацию предсказаний на тестовой выборке...")

# Берем тестовый датасет (или его часть для теста, например, первые 100 примеров)
# Если ресурсов мало, раскомментируйте срез: [:100]
test_subset = dataset["test"] # .select(range(100))

true_labels = []
pred_labels = []

# Словарь для нормализации ответов (если модель выдаст лишние пробелы или регистр)
# Адаптируйте под ваши лейблы в CSV!
label_map = {
    "POSITIVE": "POSITIVE",
    "NEGATIVE": "NEGATIVE",
    "NEUTRAL": "NEUTRAL",
    # Добавьте вариации, если модель "галлюцинирует" (например, "Positive")
}

for item in tqdm(test_subset):
    text = item["text"]
    true_label = str(item["label"]).upper() # Приводим к верхнему регистру

    # Предсказание
    raw_pred = predict_sentiment(text)

    # Очистка предсказания
    clean_pred = raw_pred.upper().strip()
    # Если модель выдала что-то странное, помечаем как ERROR или приводим к ближайшему
    if clean_pred not in label_map:
        clean_pred = "NEUTRAL" # Fallback стратегия (или сохраните как 'UNKNOWN')

    true_labels.append(true_label)
    pred_labels.append(clean_pred)

In [ ]:
plt.figure(figsize=(10, 8))
# Получаем уникальные классы из данных
labels = sorted(list(set(true_labels + pred_labels)))

cm = confusion_matrix(true_labels, pred_labels, labels=labels)

# Отрисовка с Seaborn
sns.set(style="whitegrid", font_scale=1.2)
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                 xticklabels=labels, yticklabels=labels)

plt.title('Матрица ошибок (Confusion Matrix)', fontsize=16, pad=20)
plt.xlabel('Предсказано моделью', fontsize=14)
plt.ylabel('Истинные метки', fontsize=14)
plt.tight_layout()

# Сохранение для диссертации (300 DPI — требование для печати)
plt.savefig("confusion_matrix_gemma12b.png", dpi=300)
plt.show()

In [ ]:
report = classification_report(true_labels, pred_labels, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).iloc[:-1, :].T # Транспонируем для красоты

plt.figure(figsize=(10, 6))
sns.heatmap(report_df.iloc[:-3, :-1], annot=True, cmap='RdYlGn', fmt='.2f', vmin=0, vmax=1)
plt.title('Тепловая карта метрик классификации', fontsize=16)
plt.tight_layout()
plt.savefig("metrics_heatmap.png", dpi=300)
plt.show()

In [ ]:
print("\nAccuracy Score:", accuracy_score(true_labels, pred_labels))